<a href="https://colab.research.google.com/github/Tahaahmed729/flyrank-ml-internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tahaahmed729/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis (Grain): One row represents one unique (search_query, target_url) pair observed on a specific date within the training panel.

Time Window: Mid-panel training window from March 1, 2026 to March 31, 2026 (month=2026-03). The final month (2026-06) is intentionally kept sealed as the test set

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# CODE CELL 1: Setup, Token Loading & Grain / Date Verification
import os
import pandas as pd
import numpy as np
from google.colab import userdata

# 1. Load Hugging Face READ token securely [cite: 183, 184]
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ HF_TOKEN loaded from Secrets.")
except Exception:
    print("⚠️ Please set HF_TOKEN in Colab Secrets.")

# 2. Read mid-panel dataset (March 2026) [cite: 173, 187, 192]
url = "https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/data/month=2026-03/train.parquet"
df = pd.read_parquet(url, storage_options={"token": HF_TOKEN})

# 3. Print grain & date span verification
print(f"Total Rows: {len(df):,}")
print(f"Date Span: {df['date'].min()} to {df['date'].max()}")


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* Features: hist_ctr_7d (7-day rolling historical CTR), query_length_words (word count of search query), historical_clicks_30d (30-day aggregated clicks), is_branded_query (brand flag), url_depth (path depth of URL) .
* Label: target_is_high_value (Binary Flag: 1 if $\text{CTR} > 0.05$ and $\text{Clicks} > 100$, else 0).
* Context: date, query, url (Used for dataset filtering, slicing, and grain checks).
* Excluded & Reason: Excluded post-session engagement metrics like post_click_dwell_time and same-day current session CTR to prevent target leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# CODE CELL 2: Feature Engineering & Label Creation
features_df = df[['date', 'query', 'url', 'clicks', 'impressions', 'ctr']].copy()

# Features (Knowable prior to decision moment) [cite: 174, 198-203]
features_df['hist_ctr_7d'] = features_df.groupby('query')['ctr'].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean()).fillna(0)
features_df['query_length_words'] = features_df['query'].astype(str).apply(lambda x: len(x.split()))
features_df['historical_clicks_30d'] = features_df.groupby('query')['clicks'].transform(lambda x: x.shift(1).rolling(30, min_periods=1).sum()).fillna(0)
features_df['is_branded_query'] = features_df['query'].astype(str).apply(lambda q: int(any(k in q.lower() for k in ['flyrank', 'brand', 'official'])))
features_df['url_depth'] = features_df['url'].astype(str).apply(lambda u: len([p for p in u.split('/') if p]))

# Proxy Target Label
features_df['target_is_high_value'] = ((features_df['ctr'] > 0.05) & (features_df['clicks'] > 100)).astype(int)

print("✅ Feature frame engineered successfully.")
features_df[['query', 'hist_ctr_7d', 'query_length_words', 'target_is_high_value']].head()

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification executed across three key contract claims: (1) Grain uniqueness check on (date, query, url), (2) Row count and time span verification, and (3) Availability filtering using IS TRUE.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# CODE CELL 3: Verification Queries & Deliberate Leakage Trap Experiment

# Verification Query 1: Unique Grain Check [cite: 173]
duplicates = df.duplicated(subset=['date', 'query', 'url']).sum()
print(f"Fact 1 (Grain Check): Duplicates on (date, query, url) = {duplicates}")
assert duplicates == 0, "Grain contract violated!"

# Verification Query 2: Row Count & Dates [cite: 173]
print(f"Fact 2 (Dataset Span): {len(df):,} rows between {df['date'].min()} and {df['date'].max()}")

# Verification Query 3: Availability Check with IS TRUE [cite: 173, 177]
df['is_valid_impression'] = (df['impressions'] > 0) & (df['clicks'].notna())
surviving_rows = df[df['is_valid_impression'] == True]
print(f"Fact 3 (Availability Check): {len(surviving_rows):,} / {len(df):,} rows survived IS TRUE filter.")

print("\n--- Running Deliberate Leakage Experiment (The Trap) ---") # [cite: 175, 178]

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_honest = features_df[['hist_ctr_7d', 'query_length_words', 'historical_clicks_30d', 'is_branded_query', 'url_depth']]
y = features_df['target_is_high_value']

# Honest Model
clf_honest = DecisionTreeClassifier(max_depth=3).fit(X_honest, y)
honest_score = accuracy_score(y, clf_honest.predict(X_honest))
print(f"🎯 Honest Model Accuracy Score: {honest_score:.4f}")

# Springing The Trap: Add Label-Derived Leaked Feature [cite: 175]
features_df['leaked_current_ctr'] = features_df['ctr']
X_leaked = features_df[['hist_ctr_7d', 'query_length_words', 'historical_clicks_30d', 'is_branded_query', 'url_depth', 'leaked_current_ctr']]

clf_leaked = DecisionTreeClassifier(max_depth=3).fit(X_leaked, y)
leaked_score = accuracy_score(y, clf_leaked.predict(X_leaked))
print(f"🚨 LEAKED Model Accuracy Score (The Trap): {leaked_score:.4f}")

# Delete Leaked Column to Retain Honest Baseline [cite: 175, 178]
features_df.drop(columns=['leaked_current_ctr'], inplace=True)
print("✅ Leaked column permanently deleted. Retained honest score.")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* Named Limitation: This data slice is restricted to a single mid-panel month (2026-03). It cannot account for long-term seasonal traffic changes, annual holiday search spikes, or unexpected Google Search Console layout updates.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.